# Fine-tune TrOCR on `combined_dataset` (Polish handwriting) — **Kaggle / RAM-optimized + quality fixes**

Fine-tunes `microsoft/trocr-base-handwritten` on the combined labeled line crops
(`combined_dataset/` + `combined_transcribed.txt`), then evaluates on a held-out split.

This is the **Kaggle** version of the optimized notebook. All training logic is identical;
only the data input and output paths changed (no Google Drive — Kaggle uses attached
Datasets under `/kaggle/input/` and a writable `/kaggle/working/`).

**Why this version actually learns (kept from the optimized notebook):**

A. **RAM optimisations:** lazy per-batch preprocessing via a custom `data_collator`
   (no giant cached pixel array), `gradient_checkpointing` + `use_cache=False`,
   effective batch size 4 (`per_device_train_batch_size=2` × `gradient_accumulation_steps=2`),
   lighter eval (`eval_accumulation_steps=4`, `dataloader_pin_memory=False`).

B. **Quality fixes:** aspect-preserving **letterbox** before the processor (no more
   squashed 15:1 lines), train-time augmentation, `weight_decay`, `label_smoothing`,
   cosine LR + low LR + early stopping, `no_repeat_ngram_size=3`, and
   `generation_config.max_length=128`.

**Continued training:** set `RESUME_FROM_DIR` in Step 2 to a checkpoint path (e.g. an
attached dataset) to keep training a previous fine-tune.

---
**Before running (Kaggle):**
1. **Verify your phone** (profile → Settings) so GPUs unlock.
2. Right sidebar → **Settings → Accelerator → GPU T4 ×2** (16 GB, sm_75).
   Do **not** pick P100 — Kaggle's PyTorch 2.10 dropped support for the P100's
   Pascal arch (sm_60), so it errors with *"no kernel image is available"*.
   The notebook pins itself to a single T4 to avoid DataParallel.
3. Right sidebar → **Settings → Internet → ON** — required so the base model can be
   downloaded from Hugging Face the first time.
4. Right sidebar → **Add Input → Datasets** and attach a dataset that contains
   `combined_transcribed.txt` **and** either a `combined_dataset/` folder of PNGs
   or a `combined_dataset.zip`. (Create the dataset once via *Datasets → New Dataset*
   and upload those two items.) Step 1 below auto-locates them — no slug needed.

In [ ]:
import os
# Use a single GPU. On "GPU T4 x2" this avoids HF Trainer's DataParallel,
# which is fragile with predict_with_generate eval + our custom collator.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# Kaggle's image already ships torch, torchvision, accelerate, transformers and
# datasets. We only quietly ensure transformers/datasets are recent enough and
# DO NOT reinstall torch/torchvision/numba — that's what triggers the (harmless)
# RAPIDS/numba dependency-conflict warnings. If the next cells import fine, you
# can ignore any remaining pip resolver messages: they refer to cudf/cuml/dask-cuda,
# which this notebook never uses.
!pip install -q -U "transformers>=4.40" "datasets>=2.14" 2>/dev/null

import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__)

In [ ]:
# --- GPU sanity check -------------------------------------------------------
# Fails fast with a clear message if torch can't actually run a kernel on this
# GPU (the "no kernel image is available for execution on the device" error),
# which usually means a pip install replaced Kaggle's GPU-matched torch.
import torch
print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0),
          "| capability sm_%d%d" % torch.cuda.get_device_capability(0))
    # Actually launch a kernel on the GPU — this is what trips the arch mismatch.
    _x = torch.randn(1024, 1024, device="cuda")
    _ = (_x @ _x).sum().item()
    print("GPU kernel test: OK")
else:
    raise RuntimeError(
        "No GPU visible. Right sidebar -> Settings -> Accelerator -> GPU P100, "
        "and verify your phone if the GPU options are greyed out."
    )

## Step 1 — Get the data from your attached Kaggle Dataset

Kaggle mounts every attached dataset **read-only** under `/kaggle/input/<dataset-slug>/`.
You don't need to know the slug: the cell below searches `/kaggle/input` for
`combined_transcribed.txt` and the images (a `combined_dataset/` folder if present,
otherwise it extracts `combined_dataset.zip` into the writable `/kaggle/working/`).

If it raises `FileNotFoundError`, you haven't attached the dataset yet — use
**Add Input** in the right sidebar.

In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import zipfile

# Kaggle mounts attached datasets read-only under /kaggle/input/<slug>/.
# /kaggle/working is the writable, persisted output dir.
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/data")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# --- locate the transcription file ---
txt_candidates = list(INPUT_ROOT.rglob("combined_transcribed.txt"))
if not txt_candidates:
    raise FileNotFoundError(
        "combined_transcribed.txt not found under /kaggle/input. "
        "Attach your dataset via 'Add Input' (right sidebar)."
    )
TRANSCRIPTIONS_FILE = WORK_DIR / "combined_transcribed.txt"
shutil.copy(txt_candidates[0], TRANSCRIPTIONS_FILE)


def _dir_with_most_pngs(root: Path) -> Path | None:
    """Return the directory that *directly* holds the most .png files (handles
    arbitrarily nested folders like combined_dataset/combined_dataset/)."""
    pngs = list(root.rglob("*.png"))
    if not pngs:
        return None
    return Counter(p.parent for p in pngs).most_common(1)[0][0]


# --- locate the images: prefer already-unzipped PNGs, else extract the zip ---
IMAGES_DIR = _dir_with_most_pngs(INPUT_ROOT)
if IMAGES_DIR is None:
    zip_candidates = list(INPUT_ROOT.rglob("combined_dataset.zip"))
    if not zip_candidates:
        raise FileNotFoundError(
            "No .png images and no combined_dataset.zip found under /kaggle/input."
        )
    print(f"Extracting {zip_candidates[0]} ...")
    with zipfile.ZipFile(zip_candidates[0]) as zf:
        zf.extractall(WORK_DIR)
    IMAGES_DIR = _dir_with_most_pngs(WORK_DIR)
    if IMAGES_DIR is None:
        raise FileNotFoundError("Zip extracted but no .png files were found inside it.")

n_images = len(list(IMAGES_DIR.glob("*.png")))
print("Images dir:", IMAGES_DIR, "-", n_images, "images")
print("Transcriptions file:", TRANSCRIPTIONS_FILE)

## Step 2 — Config & imports

In [ ]:
from __future__ import annotations

import gc
import inspect
import logging
import random
from dataclasses import dataclass

import numpy as np
import torch
from datasets import Dataset
from PIL import Image
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

TROCR_MODEL_BASE = "microsoft/trocr-base-handwritten"

# Where the fine-tuned model gets saved. /kaggle/working is committed as the
# notebook's output, so it survives and is downloadable after the run.
OUTPUT_MODEL_DIR = Path("/kaggle/working/models/trocr_sample1000v2")

# --- Continued training -------------------------------------------------------
# Set this to a path (e.g. an attached dataset under /kaggle/input/...) containing a previously
# fine-tuned TrOCR checkpoint (the directory you got from `trainer.save_model`,
# i.e. the same shape as OUTPUT_MODEL_DIR). The notebook will then start from
# *those* weights instead of `microsoft/trocr-base-handwritten`.
#
# Set to `None` for a fresh fine-tune from the public base model.
RESUME_FROM_DIR = None  # e.g. Path("/kaggle/input/my-trocr-checkpoint")
# -----------------------------------------------------------------------------

# Hyperparameters — effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS = 4.
EPOCHS = 25                       # was 10 — early stopping cuts it short if needed.
BATCH_SIZE = 2                    # per-device micro-batch.
GRAD_ACCUM_STEPS = 2              # accumulate to keep effective batch of 4.
# Lower LR than the original 5e-5: with augmentation + longer training + early
# stopping, slower learning gives a more stable trajectory and better CER.
LEARNING_RATE = 1e-5 if RESUME_FROM_DIR is not None else 2e-5
WEIGHT_DECAY = 0.01               # mild regularization (was 0).
LABEL_SMOOTHING = 0.1             # smoother target distribution helps small datasets.
TEST_SIZE = 0.15
SEED = 42
MAX_TARGET_LENGTH = 128
PRINT_TEST_EXAMPLES = 10
EARLY_STOP_PATIENCE = 5           # epochs of no eval-CER improvement before stopping.
USE_AUGMENTATION = True           # train-time perspective/jitter/blur (eval is unchanged).

# Number of dataloader workers. 2 is a good default on Colab; set to 0 if you ever
# see "DataLoader worker is killed" / shared-memory errors.
NUM_WORKERS = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## Step 3 — Load samples and build train/test splits

`sample_1000_transcribed_v2.txt` is tab-separated `image_name<TAB>text`.
Lines with an empty transcription (blank/illegible crops) are skipped.

In [ ]:
def _levenshtein_distance(a: str, b: str) -> int:
    """Compute Levenshtein distance with dynamic programming."""
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            cost = 0 if ca == cb else 1
            curr.append(min(
                curr[j - 1] + 1,      # insertion
                prev[j] + 1,          # deletion
                prev[j - 1] + cost,   # substitution
            ))
        prev = curr
    return prev[-1]


def _character_error_rate(refs: list[str], preds: list[str]) -> float:
    total_dist = 0
    total_chars = 0
    for ref, pred in zip(refs, preds):
        total_dist += _levenshtein_distance(ref, pred)
        total_chars += max(1, len(ref))
    return total_dist / total_chars


def load_samples(images_dir: Path, transcription_file: Path) -> list[dict[str, str]]:
    """Load labeled samples and validate image existence."""
    if not images_dir.exists():
        raise FileNotFoundError(f"Images directory not found: {images_dir}")
    if not transcription_file.exists():
        raise FileNotFoundError(f"Transcription file not found: {transcription_file}")

    records: list[dict[str, str]] = []
    skipped_empty = 0
    with open(transcription_file, "r", encoding="utf-8") as f:
        for idx, raw_line in enumerate(f, start=1):
            line = raw_line.rstrip("\n")
            if not line.strip():
                continue
            if "\t" not in line:
                raise ValueError(f"Line {idx} in {transcription_file} is not tab-separated.")

            image_name, text = line.split("\t", 1)
            if not text.strip():
                skipped_empty += 1
                continue

            image_path = images_dir / image_name
            if not image_path.exists():
                raise FileNotFoundError(f"Missing image for line {idx}: {image_name}")

            records.append({"image_path": str(image_path), "text": text})

    if not records:
        raise ValueError("No valid samples were loaded.")

    logger.info(
        "Loaded %d labeled samples (%d skipped: empty transcription).",
        len(records), skipped_empty,
    )
    return records


def build_datasets(records: list[dict[str, str]], test_size: float, seed: int) -> tuple[Dataset, Dataset]:
    """Create train/test HF datasets."""
    dataset = Dataset.from_list(records)
    split = dataset.train_test_split(test_size=test_size, seed=seed, shuffle=True)
    train_ds = split["train"]
    test_ds = split["test"]
    logger.info("Split dataset: %d train / %d test", len(train_ds), len(test_ds))
    return train_ds, test_ds


records = load_samples(IMAGES_DIR, TRANSCRIPTIONS_FILE)
train_raw, test_raw = build_datasets(records, test_size=TEST_SIZE, seed=SEED)

## Step 4 — Load model & processor

In [ ]:
# Resume from a previously fine-tuned model on Drive if RESUME_FROM_DIR is set,
# otherwise start from the public TrOCR base. The processor (image preprocessor +
# tokenizer) is loaded from the same source — it stays identical to the base
# anyway when the checkpoint was saved with `processor.save_pretrained(...)`.
load_source = str(RESUME_FROM_DIR) if RESUME_FROM_DIR is not None else TROCR_MODEL_BASE
print("Loading model+processor from:", load_source)

processor = TrOCRProcessor.from_pretrained(load_source)
model = VisionEncoderDecoderModel.from_pretrained(load_source)

# Ensure generation tokens are consistently configured.
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.vocab_size = model.config.decoder.vocab_size

if model.generation_config is not None:
    model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    model.generation_config.eos_token_id = processor.tokenizer.sep_token_id
    # The shipped TrOCR generation_config has max_length=20 which a) silently
    # truncates predictions at eval time, and b) clashes with predict_with_generate's
    # max_new_tokens, producing the noisy "Both max_new_tokens and max_length seem
    # to have been set" warning every batch. Align it with our target length.
    model.generation_config.max_length = MAX_TARGET_LENGTH
    # Discourages the decoder from looping on a single n-gram (the
    # "przedyjechać przedyjechać przedyjechać" failure mode).
    model.generation_config.no_repeat_ngram_size = 3

# RAM/VRAM-friendly training settings.
# - use_cache=False is required for gradient checkpointing (and unused during training anyway).
# - gradient_checkpointing recomputes activations in the backward pass, cutting GPU activation
#   memory by ~30-40% at the cost of ~20% slower training. Quality is unchanged.
model.config.use_cache = False
model.gradient_checkpointing_enable()

model.to(device)

# Free the temporary CPU copy of the model weights that from_pretrained held onto.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Model loaded on", device, "| gradient_checkpointing=ON | use_cache=False")

## Step 5 — Preprocess datasets

In [ ]:
# --- LAZY PREPROCESSING + ASPECT-AWARE LETTERBOX -----------------------------
#
# The original notebook ran `dataset.map(..., batched=True)` with
# `pixel_values.tolist()` for every image — that briefly held ~4–8 GB of nested
# Python floats in CPU RAM and is what was crashing Colab.
#
# Equally important: TrOCR images of single handwritten lines are typically
# panoramic (aspect ratios of 10:1 to 20:1). The default TrOCRProcessor squashes
# them into 384×384 *without preserving the aspect ratio*, which deforms letters
# beyond recognition — the encoder gives up, the decoder ignores it and starts
# hallucinating frequent training-set fragments. To fix that we letterbox the
# image (white-pad to a square preserving aspect) BEFORE handing it to the
# processor, which then resizes the already-square image cleanly.
#
# We also add light train-time augmentation (perspective + brightness/contrast
# + tiny blur) — at 1000 samples augmentation is one of the strongest defences
# against the overfitting we were seeing (train loss 7.6→1.35 while eval CER
# was flat at ~0.85). Eval is run on un-augmented, only-letterboxed images.
# -----------------------------------------------------------------------------

from torchvision import transforms as T


def letterbox_to_square(img: Image.Image, fill: int = 255) -> Image.Image:
    """Pad a (W, H) PIL image with white to a square, keeping aspect ratio."""
    w, h = img.size
    side = max(w, h)
    if w == h:
        return img
    canvas = Image.new("RGB", (side, side), (fill, fill, fill))
    # Left-align horizontally, center vertically: keeps the leading character
    # at a consistent position (helps a fixed-grid ViT encoder).
    canvas.paste(img, (0, (side - h) // 2))
    return canvas


# Train-time visual augmentation — applied on the *letterboxed* PIL image
# before the TrOCRProcessor. Conservative settings; nothing that would change
# the meaning of a character (no flips, no big rotations).
_train_augment = T.Compose([
    T.RandomApply([T.RandomPerspective(distortion_scale=0.15, p=1.0, fill=255)], p=0.5),
    T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2)], p=0.5),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.8))], p=0.3),
    T.RandomApply([T.RandomAffine(degrees=2, translate=(0.01, 0.02), fill=255)], p=0.3),
])


@dataclass
class TrOCRCollator:
    """Read images, optionally augment, letterbox, tokenize, build a batch.

    We emit *both* `labels` (for loss/metrics) and `decoder_input_ids` (the
    teacher-forcing inputs, i.e. labels shifted right with a BOS token in
    front). The pre-shifted decoder inputs are required when training with
    `label_smoothing_factor > 0`: in that case the HF Trainer pops `labels`
    out of the model call before computing loss externally, which means the
    encoder-decoder no longer sees `labels` and would otherwise raise
    `ValueError: You have to specify either decoder_input_ids or
    decoder_inputs_embeds`.
    """

    processor: TrOCRProcessor
    max_target_length: int
    augment: bool = False

    def __call__(self, features: list[dict[str, str]]) -> dict[str, torch.Tensor]:
        pil_images: list[Image.Image] = []
        for f in features:
            img = Image.open(f["image_path"]).convert("RGB")
            img = letterbox_to_square(img, fill=255)
            if self.augment:
                img = _train_augment(img)
            pil_images.append(img)

        pixel_values = self.processor(images=pil_images, return_tensors="pt").pixel_values

        # `padding="longest"` pads only to the longest sequence in the current
        # batch (loss masks pad anyway), so we save tokens vs. padding to 128.
        tokenized = self.processor.tokenizer(
            [f["text"] for f in features],
            padding="longest",
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt",
        )
        input_ids = tokenized.input_ids
        pad_id = self.processor.tokenizer.pad_token_id
        # TrOCR uses the tokenizer's CLS token as the decoder start token
        # (we set this in Step 4 on model.config.decoder_start_token_id).
        bos_id = self.processor.tokenizer.cls_token_id

        # Shift right: decoder_input_ids[:, 0] = BOS, decoder_input_ids[:, t] = input_ids[:, t-1].
        decoder_input_ids = input_ids.new_full(input_ids.shape, pad_id)
        decoder_input_ids[:, 0] = bos_id
        decoder_input_ids[:, 1:] = input_ids[:, :-1]

        # Labels are the *un-shifted* targets, with pad masked out for the loss.
        labels = input_ids.masked_fill(input_ids == pad_id, -100)

        return {
            "pixel_values": pixel_values,
            "labels": labels,
            "decoder_input_ids": decoder_input_ids,
        }


# No `.map()`, no Arrow cache: datasets stay as `{image_path, text}` rows.
train_ds = train_raw
test_ds = test_raw

train_data_collator = TrOCRCollator(
    processor=processor,
    max_target_length=MAX_TARGET_LENGTH,
    augment=USE_AUGMENTATION,
)
eval_data_collator = TrOCRCollator(
    processor=processor,
    max_target_length=MAX_TARGET_LENGTH,
    augment=False,
)

print(
    f"Train: {len(train_ds)}  |  Test: {len(test_ds)}  |  "
    f"preprocessing: lazy + letterbox  |  augment(train)={USE_AUGMENTATION}"
)

## Step 6 — Metrics, training args & trainer

In [ ]:
def build_compute_metrics(processor: TrOCRProcessor):
    """Build CER + exact-match metrics for Trainer."""

    def _compute_metrics(eval_preds):
        pred_ids, label_ids = eval_preds
        if isinstance(pred_ids, tuple):
            pred_ids = pred_ids[0]

        # Newer transformers pads variable-length predictions across eval batches with -100;
        # replace those before decoding or the tokenizer raises an OverflowError.
        pred_ids = np.where(pred_ids != -100, pred_ids, processor.tokenizer.pad_token_id)

        # Replace ignore index to decode labels
        labels = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)

        pred_texts = processor.batch_decode(pred_ids, skip_special_tokens=True)
        label_texts = processor.batch_decode(labels, skip_special_tokens=True)

        pred_texts = [p.strip() for p in pred_texts]
        label_texts = [l.strip() for l in label_texts]

        cer = _character_error_rate(label_texts, pred_texts)
        exact_match = sum(p == l for p, l in zip(pred_texts, label_texts)) / max(1, len(label_texts))
        return {"cer": cer, "exact_match": exact_match}

    return _compute_metrics


from transformers import EarlyStoppingCallback


def build_training_args(output_dir: Path, use_fp16: bool) -> Seq2SeqTrainingArguments:
    """Build Seq2SeqTrainingArguments across transformers API variants."""
    common_kwargs = {
        "output_dir": str(output_dir),
        "per_device_train_batch_size": BATCH_SIZE,
        "per_device_eval_batch_size": BATCH_SIZE,
        # Effective batch = BATCH_SIZE * GRAD_ACCUM_STEPS = 4.
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "learning_rate": LEARNING_RATE,
        "num_train_epochs": EPOCHS,
        # Cosine schedule + short warmup is friendlier to a fine-tune than the
        # default linear schedule, especially with a low LR.
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing_factor": LABEL_SMOOTHING,
        "save_strategy": "epoch",
        "logging_strategy": "steps",
        "logging_steps": 10,
        "predict_with_generate": True,
        "generation_max_length": MAX_TARGET_LENGTH,
        "fp16": use_fp16,
        # Disk-saving (prevents the Kaggle "tried to use more disk space than is
        # available" crash): each full checkpoint also stores the optimizer state
        # (~2.6 GB) + scheduler + RNG on top of the ~1.3 GB weights. save_only_model
        # drops all but the weights, and save_total_limit=1 keeps a single checkpoint,
        # cutting peak checkpoint disk from ~12 GB to ~2.6 GB. We run to completion in
        # one go (no mid-run resume), and load_best_model_at_end still restores the
        # best weights at the end.
        "save_only_model": True,
        "save_total_limit": 1,
        "load_best_model_at_end": True,
        "metric_for_best_model": "cer",
        "greater_is_better": False,
        "report_to": "none",
        # --- RAM / VRAM tuning -------------------------------------------------
        "eval_accumulation_steps": 4,
        "remove_unused_columns": False,
        "dataloader_num_workers": NUM_WORKERS,
        "dataloader_pin_memory": False,
    }

    params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    if "evaluation_strategy" in params:
        common_kwargs["evaluation_strategy"] = "epoch"
    elif "eval_strategy" in params:
        common_kwargs["eval_strategy"] = "epoch"
    else:
        logger.warning("No eval strategy arg found; falling back to no periodic evaluation.")

    return Seq2SeqTrainingArguments(**common_kwargs)


class _TwoCollatorTrainer(Seq2SeqTrainer):
    """Seq2SeqTrainer that uses a different data_collator for eval (no augmentation)."""

    def __init__(self, *args, eval_data_collator=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._eval_data_collator = eval_data_collator

    def get_eval_dataloader(self, eval_dataset=None):
        if self._eval_data_collator is None:
            return super().get_eval_dataloader(eval_dataset)
        original = self.data_collator
        self.data_collator = self._eval_data_collator
        try:
            return super().get_eval_dataloader(eval_dataset)
        finally:
            self.data_collator = original


def build_trainer(
    model: VisionEncoderDecoderModel,
    training_args: Seq2SeqTrainingArguments,
    train_ds: Dataset,
    test_ds: Dataset,
    processor: TrOCRProcessor,
    train_data_collator,
    eval_data_collator,
) -> Seq2SeqTrainer:
    """Build a Seq2SeqTrainer with separate collators for train/eval."""
    kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_ds,
        "eval_dataset": test_ds,
        "data_collator": train_data_collator,
        "eval_data_collator": eval_data_collator,
        "compute_metrics": build_compute_metrics(processor),
        "callbacks": [EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
    }

    params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    if "tokenizer" in params:
        kwargs["tokenizer"] = processor.tokenizer
    elif "processing_class" in params:
        kwargs["processing_class"] = processor
    else:
        logger.warning("No tokenizer/processing_class argument found in Seq2SeqTrainer.")

    return _TwoCollatorTrainer(**kwargs)


# Checkpoints go to an ephemeral dir (NOT /kaggle/working) so multi-GB optimizer
# states aren't committed as notebook output. The final model is saved to
# /kaggle/working in Step 8.
CHECKPOINT_DIR = Path("/kaggle/temp/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

use_fp16 = torch.cuda.is_available()
training_args = build_training_args(CHECKPOINT_DIR, use_fp16)
trainer = build_trainer(
    model, training_args, train_ds, test_ds, processor,
    train_data_collator=train_data_collator,
    eval_data_collator=eval_data_collator,
)

# Free anything left over from setup before training kicks off.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Step 7 — Train

In [ ]:
trainer.train()

In [ ]:
eval_metrics = trainer.evaluate()
print("Final metrics:", eval_metrics)

## Step 8 — Save the fine-tuned model to `/kaggle/working`

Saved under `/kaggle/working/models/...`, which Kaggle commits as the notebook's
output — download it from the **Output** tab, or *Save Version* to keep it.

In [ ]:
OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_MODEL_DIR))
processor.save_pretrained(str(OUTPUT_MODEL_DIR))
print("Saved model to", OUTPUT_MODEL_DIR)

## Step 9 — Qualitative test on held-out examples

In [ ]:
def run_qualitative_test(
    model: VisionEncoderDecoderModel,
    processor: TrOCRProcessor,
    test_records: list[dict[str, str]],
    num_examples: int,
) -> None:
    """Print sample predictions from the test split."""
    if not test_records:
        print("No test samples available for qualitative test.")
        return

    # Re-enable KV cache for fast autoregressive decoding (was disabled for
    # gradient checkpointing during training). This only affects inference speed.
    model.config.use_cache = True
    model.eval()
    device = model.device

    subset = test_records[: min(num_examples, len(test_records))]
    for i, record in enumerate(subset, start=1):
        image = Image.open(record["image_path"]).convert("RGB")
        pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
        with torch.no_grad():
            generated_ids = model.generate(pixel_values, max_new_tokens=128)
        pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        truth = record["text"].strip()
        print(f"[{i}] GT: {truth}")
        print(f"    PR: {pred}")


test_records = [test_raw[i] for i in range(len(test_raw))]
run_qualitative_test(trainer.model, processor, test_records, PRINT_TEST_EXAMPLES)